# NQX-Core Intro
## KV-cache quantization with NautilusQuant

This notebook demonstrates the NQX-Core emulator — a specialized processor for KV-cache compression using golden-ratio rotations.

**No PyTorch required.** Only numpy.

In [ ]:
import numpy as np
from nqx.constants import NQXConfig
from nqx.cpu import NQXCore

## 1. What is KV-cache?

In LLM inference, the Key-Value cache stores intermediate attention matrices. For a 70B model with 32K context, this can be **several GB** — often the memory bottleneck.

NautilusQuant compresses KV-cache by:
1. **Rotating** vectors with 3 Givens layers (golden-angle)
2. **Polar transform**: (r, θ) pairwise
3. **Quantizing** radius to 3 bits + 1 sign bit
→ **4x compression** with minimal accuracy loss

In [ ]:
# Generate 16 random vectors dim=128
rng = np.random.default_rng(0)
x = rng.standard_normal((16, 128)).astype(np.float32)
print(f"Input shape: {x.shape}, dtype: {x.dtype}")
print(f"Range: [{x.min():.3f}, {x.max():.3f}]")

In [ ]:
# Encode and decode
cfg = NQXConfig(dim=128, bits=3)
core = NQXCore(cfg)

enc = core.encode(x)
dec = core.decode(enc)

# Check compression
raw_bytes = x.size * 2  # FP16
packed_bytes = len(enc.packed_bytes)
ratio = raw_bytes / packed_bytes
print(f"Raw: {raw_bytes} bytes → Packed: {packed_bytes} bytes")
print(f"Compression: {ratio:.2f}x (target: 4.00x)")

In [ ]:
# Check reconstruction error
rmse = float(np.sqrt(((x - dec.reconstructed) ** 2).mean()))
print(f"Roundtrip RMSE: {rmse:.4f}")
print(f"Cycles: {enc.cycles} (enc) + {dec.cycles} (dec)")
print(f"Energy: {enc.energy_nj:.2f} nJ")

In [ ]:
# Visualize: first 3 vectors, first 16 dims before/after
try:
    import matplotlib.pyplot as plt
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].plot(x[:3].T, 'o-')
    axes[0].set_title('Original')
    axes[1].plot(dec.reconstructed[:3].T, 'o-')
    axes[1].set_title('Reconstructed')
    plt.tight_layout()
    plt.show()
except ImportError:
    print("matplotlib not available — skipping visualization")
    err = np.abs(x - dec.reconstructed).mean(axis=0)
    print(f"Mean abs error per dimension: {err[0]:.4f} .. {err[-1]:.4f}")